# LELA60331 Week 8 Seminar: Training a softmax classifier

Last week we saw how a **softmax** classifier turns a z value for each class into a probability for each class. This week we **train** one: first by working through a single update by hand, and then by training a classifier on the simulated three-class sentiment data from last week. Finally we see how to calculate precision and recall when there are more than two classes.

| Part | Topic |
|---|---|
| 1 | The forward pass |
| 2 | The backward pass |
| 3 | Training on the simulated data |
| 4 | Precision and recall for several classes |

**Lectures:** this sheet goes with this week's two lectures, *Neural Networks 2: Training* and *Training a softmax classifier*. Parts 1 and 2 work through the same example as the softmax lecture, so you can check your numbers against the slides.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(suppress=True, precision=4)    # print numbers without scientific notation

def softmax(z):
    e = np.exp(z)
    return e / e.sum()

---
# Part 1 · The forward pass

Making a prediction from an input is called the **forward pass**.

### The bias trick

So far each class has had some weights **and** a separate bias. A common trick is to add an extra feature to every input that is always **1**. Its weight then does the job of the bias: it is added to z whatever the other features are. This means the weights and the bias can be stored together in a single matrix.

Below, `x` is one input with two features plus the extra 1. `weights` has one **row per feature** (including the extra 1) and one **column per class**, so `weights[i, k]` is the weight from feature `i` to class `k`. Then `x.dot(weights)` gives the z value of every class at once: for each class (column), the dot product of `x` with that column.

### 🏋️ Exercise 1 — A forward pass

Calculate `z`, the z value of each of the three classes, and `q`, the probability of each class.

In [ ]:
weights = np.array([[0.77, 0.63, 0.50],     # weights from feature 0 to classes 0, 1, 2
                    [0.02, 0.74, 0.22],     # weights from feature 1
                    [0.20, 0.76, 0.17]])    # weights from the extra 1 (the biases)
x = np.array([2.33, 3.72, 1.0])

z = ???
q = ???
print(z)
print(q)            # you should get [0.0488 0.898  0.0531]

Which class does the model predict?

---
# Part 2 · The backward pass

Suppose the correct class for this input is class **0**, so its one-hot label is `y = [1, 0, 0]`. The model gave class 0 a probability of only 0.05, so the weights need to change. Working out how to change them, from the error back to the weights, is called the **backward pass**.

The rule is very like the one for logistic regression in Week 4. For each class, the **error** is the predicted probability minus the correct value: `q - y`. The gradient for the weight from feature `i` to class `k` is the input value times the error:

$$\delta w_{i,k} = x_i \cdot (q_k - y_k)$$

So if the model gave a class too **much** probability (positive error), the weights feeding into it go **down**, and if it gave too **little** (negative error), they go **up**, in each case by more when the input feature was bigger.

`np.outer(x, error)` makes the matrix of all these products at once: row `i`, column `k` holds `x[i] * error[k]`. It has the same shape as `weights`.

### 🏋️ Exercise 2 — One update

1. Calculate `error`, the vector `q - y`.
2. Calculate `dw`, the matrix of gradients, with `np.outer`.
3. Update the weights with a learning rate of 0.2.
4. Recalculate the probabilities. 🔮 Before running: should the probability of class 0 go up or down?

In [ ]:
y = np.array([1, 0, 0])
lr = 0.2

error = ???
dw = ???
weights = ???
print(weights)
# you should get:
# [[1.2133 0.2115 0.4752]
#  [0.7277 0.0719 0.1805]
#  [0.3902 0.5804 0.1594]]

q_new = ???
print(q_new)

One update has moved most of the probability onto the correct class. Training repeats this, over and over, for every item in the training data.

---
# Part 3 · Training on the simulated data

Here is the simulated three-class data from last week, with its one-hot labels `y`. We add the extra column of 1s for the bias trick with `np.column_stack`, which puts arrays side by side as columns.

In [ ]:
np.random.seed(10)
positive = np.random.normal((3, 1), 0.5, size=(20, 2))       # 20 texts around (3, 1)
negative = np.random.normal((1, 3), 0.5, size=(20, 2))       # 20 texts around (1, 3)
neutral = np.concatenate([np.random.normal((1, 1), 0.5, size=(10, 2)),
                          np.random.normal((3, 3), 0.5, size=(10, 2))])
x = np.concatenate([positive, negative, neutral])            # 60 rows, 2 columns
labs = np.array([0] * 20 + [1] * 20 + [2] * 20)              # 0 = positive, 1 = negative, 2 = neutral
y = np.eye(3)[labs]                                          # one-hot labels: 60 rows, 3 columns

X = np.column_stack([x, np.ones(len(x))])                    # add a column of 1s
print(X.shape, y.shape)
print(X[:3])

To train on all 60 texts at once:

* `X.dot(weights)` gives a matrix of z values, with one **row per text** and one column per class.
* We need softmax for **each row**. The function `softmax_rows` below does this. `keepdims=True` keeps the row totals as a column, so that each row is divided by its own total.
* The **loss** is the **cross-entropy**: for each text, minus the log of the probability given to the correct class, averaged over all texts. `np.sum(y * q, axis=1)` picks out that probability for each text, because `y` is 1 for the correct class and 0 elsewhere.
* The gradient for all the weights is `X.T.dot(q - y)` divided by the number of texts: the matrix version of Part 2, summed over all the texts.

In [ ]:
def softmax_rows(z):
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

print(softmax_rows(np.array([[2.0, 1.0, 0.1],
                             [0.0, 0.0, 0.0]])))

### 🏋️ Exercise 3 — The training loop

Complete the training loop:

1. `z`: the z values for every text and class
2. `q`: the probabilities, using `softmax_rows`
3. `dw`: the gradient for every weight
4. update the weights

In [ ]:
np.random.seed(10)
num_samples, num_features = X.shape
num_classes = 3
weights = np.random.rand(num_features, num_classes)
n_iters = 1000
lr = 0.2
ce_loss = []

for i in range(n_iters):
    z = ???
    q = ???
    loss = np.mean(-np.log2(np.sum(y * q, axis=1)))
    ce_loss.append(loss)

    dw = ???
    weights = ???

plt.plot(range(1, n_iters), ce_loss[1:])
plt.xlabel("number of epochs")
plt.ylabel("loss")
plt.show()

### 🏋️ Exercise 4 — Predictions and accuracy

1. Calculate `q`, the probabilities for every text, using the trained weights.
2. Set `y_pred` to the predicted class of every text (the position of the largest probability in each row) and `y_true` to the correct class of every text.
3. Calculate the accuracy.

In [ ]:
q = ???
y_pred = ???
y_true = ???
print(y_pred)
print("accuracy:", ???)

Look at which texts are misclassified, and compare them with the scatter plot from last week: they are mostly texts that lie near the border between two groups. A softmax classifier divides the space into regions with **straight** borders. That works here, because the neutral texts lie in a diagonal band between the positive and negative ones. In Week 9 we will meet a problem where straight borders are not enough.

### 🏋️ Exercise 5 — Classify new texts

Write `predict_sentiment(n_positive, n_negative)`, which returns the predicted class name for a text with the given counts. Remember that the features are **log** counts, plus the extra 1.

In [ ]:
class_names = ["positive", "negative", "neutral"]

def predict_sentiment(n_positive, n_negative):
    features = ???
    probs = ???
    return ???

print(predict_sentiment(20, 2), predict_sentiment(2, 20), predict_sentiment(3, 3))

---
# Part 4 · Precision and recall for several classes

In Week 5 we calculated precision and recall for a binary classifier, treating "positive" as the class of interest. With several classes we can do the same for **each class in turn**, treating that class as "positive" and all the others together as "negative":

* **true positives** for class *k*: texts predicted as *k* that really are *k*
* **false positives** for class *k*: texts predicted as *k* that are really something else
* **false negatives** for class *k*: texts that really are *k* but were predicted as something else

This gives a precision and a recall for each class. **Macro-averaging** then takes the plain average over the classes, so that every class counts equally, however many texts it has.

### 🏋️ Exercise 6 — Macro-averaged precision and recall

Complete `macro_precision_recall(y_pred, y_true, num_classes)`. For each class `k` it counts the true positives, false positives and false negatives with NumPy (as in Week 7), works out that class's precision and recall, and finally returns the averages. *Hint:* for the true positives, count where `y_pred == k` **and** `y_true == k`.

In [ ]:
def macro_precision_recall(y_pred, y_true, num_classes):
    precisions = []
    recalls = []
    for k in range(num_classes):
        tp = ???
        fp = ???
        fn = ???
        precisions.append(???)
        recalls.append(???)
    print("precision per class:", np.round(precisions, 3))
    print("recall per class:   ", np.round(recalls, 3))
    return float(np.mean(precisions)), float(np.mean(recalls))

print(macro_precision_recall(np.array([0, 1, 1, 2, 2, 0]), np.array([0, 1, 2, 2, 1, 1]), 3))
# per class precision [0.5, 0.5, 0.5], recall [1.0, 0.333, 0.5]; averages (0.5, 0.611)

In [ ]:
print(macro_precision_recall(y_pred, y_true, 3))

Which class has the lowest precision or recall? Does that fit with what you saw in Exercise 4?

⭐ **Challenge (optional):** retrain the model with a different learning rate (try 0.02 and 2) or a different number of iterations. How do the loss curve and the macro-averaged scores change?

---
## ✏️ Summary

This week you have:

* used the **bias trick** to store biases as weights
* carried out a **forward pass** (z values, then softmax) and a **backward pass** (error `q - y`, gradient `np.outer(x, error)`) by hand
* trained a softmax classifier on all the data at once with matrix operations and the **cross-entropy** loss
* calculated **macro-averaged** precision and recall for several classes